# 📊 Superstore End-to-End Data Analysis: Customer Intelligence & Regional Profitability

## 📝 Project Overview

This project delivers a comprehensive, business-driven analysis of the **Superstore** dataset using advanced SQL queries. The primary goal is to evaluate commercial performance, unravel customer buying behaviors, and identify key drivers of profitability versus revenue leaks across product categories and geographic regions.

---

## 🛠️ Tools & Technologies Used

* **SQL Server (T-SQL):** Primary tool for data transformation, advanced aggregations, and business logic implementation.
* **Jupyter Notebook (`ipython-sql`):** Environment for documenting execution flows and contextual markdown insights.
* **Key SQL Techniques:** Common Table Expressions (CTEs), Window Functions (`NTILE`, `OVER()`), Data Aggregations, Conditional Logic (`CASE WHEN`), Division-by-Zero Protection (`NULLIF`), and Precise Rounding (`CAST/DECIMAL`).

---

## 🚀 Data Analysis Journey

1. **Data Cleanup & Preprocessing:** Ensuring numeric precision, proper rounding, and strict customer identification parsing.
2. **RFM Segmentation:** Grouping customers into distinct behavioral segments (*Champions*, *Potential Loyalists*, *At Risk*, *Hibernating*) using score quantiles.
3. **CLV & Profitability:** Merging customer metrics with financial indicators to isolate high-value cohorts from low-margin segments.
4. **Product & Category Performance:** Evaluating profit margins, sales volumes, and discount impacts across Sub-Categories.
5. **Geographic & Regional Analysis:** Mapping revenue against profit margins at the Region, State, and City levels to pinpoint top-performing markets and loss-making territories.

---

## **💡 Strategic Business Recommendations**

**1. Profitability & Pricing Optimization:**

* **Reevaluate Discount Strategy:** Restructure the discounting model—specifically capping high discounts on unprofitable sub-categories—to protect net margins.
* **Optimize Office Supplies Category:** Conduct a granular product-level review of Office Supplies to trim loss-making SKUs rather than discontinuing the entire category.

**2. Supply Chain & Inventory Planning:**

* **Q4 Inventory Readiness:** Scale up inventory allocation and supply chain capacity ahead of **Q4**, leveraging its status as the highest-volume sales quarter.
* **Logistics & Delay Audit:** Investigate root causes behind the high delay rates (78%-82%) across regions to streamline carrier performance and lower operational friction.

**3. Category & Regional Growth:**

* **Capitalize on Technology:** Increase marketing and inventory investment in **Technology**, as it drives ~50% of total profits.
* **Targeted Regional Ads:** Expand regional ad campaigns for Technology products across Central, East, and South regions to capture unmet demand.

**4. Tailored RFM Marketing Strategies:**

* **Champions:** Launch VIP reward programs and exclusive perks to maximize retention and Lifetime Value (CLV).
* **Potential Loyalists:** Deploy cross-selling and tailored upsell bundles to encourage higher order frequency.
* **At Risk:** Implement win-back campaigns and personalized incentive offers before customers fully churn.
* **Hibernating:** Test low-cost reactivation email sequences to re-engage lapsed buyers efficiently.

### importing the data

In [8]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
                                                                                                                                         
%sql mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes

---
## **1. 📌 Executive Overview & Core KPIs**

In [12]:
%%sql
select 
/*Financial KPIs*/
cast(round(SUM(sales),1)as decimal(18,2)) total_sales,
cast(round(SUM(profit),1)as decimal(18,2)) total_profit,
cast(round(AVG(Discount)*100,1)as decimal(18,2)) avg_discount,
cast(round((SUM(profit)/SUM(sales))*100,1)as decimal(18,2)) profit_margin,
/*Operational & Volume KPIs*/
count(distinct order_id) total_orders,
sum(quantity) total_quantity,
cast(round((SUM(sales)/count(distinct order_id)),1)as decimal(18,2)) AOV,
/* Product Level Overview*/
count(distinct Product_ID) total_Product
from Superstore;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


total_sales,total_profit,avg_discount,profit_margin,total_orders,total_quantity,AOV,total_Product
2297200.90,286397.00,15.60,12.50,5009,37873,458.60,1862


In [13]:
%%sql
select top(1)
 year_order,
 coalesce(round(cast(current_year_sales - previous_year_sales as float )/previous_year_sales*100,1),0) as  sales_year_change_percentage,
 cast(round(coalesce(((current_year_profit - previous_year_profit )/previous_year_profit*100),0),1) as decimal(18,2))as  profit_year_change_percentage
from(
	select 
	year(Order_Date) year_order,
	round(cast(sum(Sales)as float),1) current_year_sales,
	lag(round(cast(sum(Sales)as float),1))over(order by year(Order_Date)) previous_year_sales,
	round(cast(sum(profit)as float),1) current_year_profit,
	lag(round(cast(sum(profit)as float),1))over(order by year(Order_Date)) previous_year_profit
	from Superstore
	group by year(Order_Date)
)t
order by year_order desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


year_order,sales_year_change_percentage,profit_year_change_percentage
2017,20.4,14.20


### **💡 Executive Summary & Key Performance Indicators (KPIs)**

**Financial Overview:**

* **Total Sales Revenue:** **$2.30M** with a **20.4% YoY Sales Growth**
* **Total Net Profit:** **$286K**, generating a **12.5% Profit Margin** and a **14.2% YoY Profit Growth**
* **Average Discount:** Maintained at **15.6%** across all transactions

**Operational & Sales Metrics:**

* **Total Orders:** **5,009 orders** fulfilled across **1,862 unique products**
* **Total Units Sold:** **37K units**
* **Average Order Value (AOV):** **$458.6** per transaction

---

## **2. 📌 Time Series & Seasonality Analysis**

Evaluating monthly and yearly sales trends to identify seasonality peaks and evaluate revenue dynamics over time.

### **MOM sales analysis**

In [28]:
%%sql
select
month_order,
Month_name,
coalesce(round(cast(current_month_sales - previous_month_sales as float ),1),0) as Sales_change,
coalesce(round(cast(current_month_sales - previous_month_sales as float )/previous_month_sales*100,1),0) as  [Sales_change_%],
coalesce(round(cast(current_month_profit - previous_month_profit as float ),1),0) as Profit_change,
coalesce(round(cast(current_month_profit - previous_month_profit as float )/previous_month_profit*100,1),0) as  [Profit_change_%]
from
(
	select 
	month(order_date)as month_order,
    format(order_date,'MMM') Month_name,
	round(cast(sum(sales)as float),1) as current_month_sales ,
	round(cast(lag(sum(sales))over(order by month(order_date))as float ),1)as previous_month_sales,
    round(cast(sum(Profit)as float),1) as current_month_profit ,
    round(cast(lag(sum(profit))over(order by month(order_date))as float ),1)as previous_month_profit
	from Superstore
	group by
    month(order_date),
    format(order_date,'MMM')
)t 
order by month_order desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


month_order,Month_name,Sales_change,Sales_change_%,Profit_change,Profit_change_%
12,Dec,-27167.6,-7.7,7900.8,22.3
11,Nov,152138.1,75.9,3684.4,11.6
10,Oct,-107326.9,-34.9,-5073.5,-13.8
9,Sep,148605.8,93.4,15080.6,69.3
8,Aug,11806.0,8.0,7944.2,57.4
7,Jul,-5480.6,-3.6,-7453.1,-35.0
6,Jun,-2310.1,-1.5,-1125.5,-5.0
5,May,17266.7,12.5,10823.9,93.4
4,Apr,-67243.4,-32.8,-17007.3,-59.5
3,Mar,145254.2,243.1,18300.1,177.8


### **YOY Analysis**

In [23]:
%%sql
select
 year_order,
 coalesce(round(cast((current_year_sales - previous_year_sales)as float ),1),0) Sales_change,
 coalesce(round(cast(current_year_sales - previous_year_sales as float )/previous_year_sales*100,1),0) as  [Sales_change_%],
coalesce(round(cast((current_year_profit - previous_year_profit)as float ),1),0) Profit_change,
coalesce(round(cast(current_year_profit - previous_year_profit as float )/previous_year_profit*100,1),0) as  [Profit_change_%]
from(
	select 
	year(Order_Date) year_order,
	round(cast(sum(Sales)as float),1) current_year_sales,
	lag(round(cast(sum(Sales)as float),1))over(order by year(Order_Date)) previous_year_sales,
    round(cast(sum(profit)as float),1) current_year_profit,
    lag(round(cast(sum(profit)as float),1))over(order by year(Order_Date)) previous_year_profit
	from Superstore
	group by year(Order_Date)
)t
order by year_order desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


year_order,Sales_change,Sales_change_%,Profit_change,Profit_change_%
2017,124009.7,20.4,11644.1,14.2
2016,138673.1,29.5,20176.6,32.7
2015,-13715.0,-2.8,12074.6,24.4
2014,0.0,0.0,0.0,0.0


### **Seasonality Sales Analysis**

In [16]:
%%sql
select 
	YEAR(order_date) year_date,
	'Q'+DATENAME(quarter,order_date) quarter,
	round(cast(SUM(sales)as float),1) total_sales
	from Superstore
	group by 
	YEAR(order_date) ,
	DATENAME(quarter,order_date)
    order by 
    YEAR(order_date) ,
    round(cast(SUM(sales)as float),1) desc
;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


year_date,quarter,total_sales
2014,Q4,179627.7
2014,Q3,143633.2
2014,Q2,86538.8
2014,Q1,74447.8
2015,Q4,182297.0
2015,Q3,130259.6
2015,Q2,89124.2
2015,Q1,68851.7
2016,Q4,236098.8
2016,Q3,143787.4


#### 💡 Quarterly Seasonality Trend:
Sales consistently increase throughout the year for all 4 years. They start at their **lowest in Q1** and reach their **peak in Q4** due to holiday shopping

---

## **3. 📌 Product & Category Performance**

Analyzing revenue distribution, profitability, and discount impacts across Sub-Categories and Products to identify key profit drivers and loss-making items.

#### **category Analysis**

In [11]:
%%sql
select 
Category,
round(cast(SUM(sales) as DECIMAL(18,2)),1) total_sales,
round(cast(SUM(Profit)as DECIMAL(18,2)),1) total_profit,
round(cast((SUM(Profit)/SUM(sales))*100  as DECIMAL(18,2)),1) as profit_margin,
round(cast((SUM(Sales)/SUM(Quantity))  as DECIMAL(18,2)),1) as sales_per_unit,
round(cast((SUM(Profit)/SUM(Quantity)) as DECIMAL(18,2)),1) as profit_per_unit,
round(cast((SUM(Profit)/(select sum(profit) from Superstore))*100  as DECIMAL(18,2)),1)  Category_profit_contribution,
SUM(quantity) total_quantity,
round(cast(AVG(discount)*100 as DECIMAL(18,2)),1) avg_discount
from Superstore
group by Category
order by round(cast(SUM(sales) as DECIMAL(18,2)),1) desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


Category,total_sales,total_profit,profit_margin,sales_per_unit,profit_per_unit,Category_profit_contribution,total_quantity,avg_discount
Technology,836154.00,145455.00,17.40,120.50,21.00,50.80,6939,13.20
Furniture,741999.80,18451.30,2.50,92.40,2.30,6.40,8028,17.40
Office Supplies,719047.00,122490.80,17.00,31.40,5.40,42.80,22906,15.70


#### **💡 Category Performance Insights**

* **Technology (Top Profit Driver):** Generates the highest overall performance, contributing **50.8%** of the total profit.
* **Office Supplies (High Efficiency & Volume):** Leads in total quantity sold and achieves a strong **42.8%** of total profit, despite having the lowest sales revenue.
* **Furniture (Profit Margin Leak):** Ranks 2nd in total sales but performs worst in profit—contributing only **6.4%** to total profit—mainly driven by the highest average discounts.

#### **Sup_category Analysis**

In [21]:
%%sql
select 
top(5)
Sub_Category,
round(cast(SUM(sales) as DECIMAL(18,2)),1) total_sales,
round(cast(SUM(Profit) as DECIMAL(18,2)),1) total_profit,
round(cast((SUM(Profit)/SUM(sales))*100 as DECIMAL(18,2)),1) as profit_margin,
round(cast((SUM(Sales)/SUM(Quantity))  as DECIMAL(18,2)),1) as sales_per_unit,
round(cast((SUM(Profit)/SUM(Quantity)) as DECIMAL(18,2)),1) as profit_per_unit,
round(cast((SUM(Profit)/(select sum(profit) from Superstore))*100  as DECIMAL(18,2)),1) profit_contribution,
SUM(quantity) total_quantity,
round(cast(AVG(discount)*100 as DECIMAL(18,2)),1) avg_discount
from Superstore
group by
Sub_Category
order by round(cast(SUM(sales) as DECIMAL(18,2)),1) asc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


Sub_Category,total_sales,total_profit,profit_margin,sales_per_unit,profit_per_unit,profit_contribution,total_quantity,avg_discount
Fasteners,3024.30,949.50,31.40,3.30,1.00,0.30,914,8.20
Labels,12486.30,5546.30,44.40,8.90,4.00,1.90,1400,6.90
Envelopes,16476.40,6964.20,42.30,18.20,7.70,2.40,906,8.00
Art,27118.80,6527.80,24.10,9.00,2.20,2.30,3000,7.50
Supplies,46673.50,-1189.10,-2.60,72.10,-1.80,-0.40,647,7.70


#### **💡 Sub-Category Performance Insights**

**Top 5 in Sales Highlights:**

* **Phones (Technology):** Strongest overall performer—ranks **#1 in Total Sales** and **#2 in Profit Contribution (15.5%)**.
* **Tables (Furniture):** Revenue vs. Profit mismatch—ranks **#3 in Total Sales** but is severely unprofitable, contributing **-6.2% to total profit** (a major profit leak).

**Bottom 5 in Sales Highlights:**

* **Low-Price Office Supplies:** Categories like *Fasteners* and *Labels* drive low sales volume due to low unit prices (**$3 – $70**) and moderate discount ranges (**6.9% – 8.2%**).
* **Supplies (Office Supplies):** Despite having the highest unit price in this group (**~$72**), it generates net losses with a **-0.4% profit contribution**.

---

## 4. Customer Behavior & Loyalty Analysis (RFM & CLV)

Leveraging the `v_Customer_Loyalty` SQL view to analyze customer retention, purchase frequency, and Lifetime Value (CLV) distribution.

#### **Customer Loyalty & Order Recency Analysis: Ranking Customers by Average Days Between Purchases**

In [48]:
%%sql
-- step 1: Removing duplicate rows per Order ID
with distict_orders as
(
	select
		Order_ID,
		Customer_ID,
		Customer_Name,
		Order_Date
	from Superstore
	group by 
		Order_ID,
		Customer_ID,
		Customer_Name,
		Order_Date
),
-- step 2: Calculating days difference between successive orders
order_differences as
(
select 
customer_id ,
Customer_Name,
DATEDIFF(day,order_date,lead(order_date)over (partition by customer_id order by order_date,order_id)) as DaysUntilNextOrder
from distict_orders
)
-- step 3:Computing Average Metrics and Customer Ranking
select 
customer_id,
Customer_Name,
avg (DaysUntilNextOrder) as avg_days,
rank()over(order by case when avg(DaysUntilNextOrder)is null then 1 else 0 end,avg(DaysUntilNextOrder) asc ) as rank_avg
from order_differences
group by 
customer_id  ,
Customer_Name;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


customer_id,Customer_Name,avg_days,rank_avg
GR-14560,Georgia Rosenberg,2,1
CD-12280,Christina DeMoss,13,2
NB-18580,Nicole Brennan,26,3
CM-12235,Chris McAfee,55,4
MH-18115,Mick Hernandez,61,5
JJ-15760,Joel Jenkins,64,6
NM-18520,Neoma Murray,66,7
BD-11770,Bryan Davis,66,7
SJ-20125,Sanjit Jacobs,66,7
CJ-12010,Caroline Jumper,67,10


#### **💡 Customer Loyalty & Purchase Frequency Insights**

* **High-Frequency Buyers:** Top loyal customers, such as *Georgia Rosenberg*, show an average reorder interval of just **2 days**, indicating high brand engagement and repeat purchasing.
* **One-Time Buyers (`NULL` Avg Days):** Customers with `NULL` or `None` values for average days between orders represent **single-purchase customers** who have not yet made a repeat transaction, making them prime targets for re-engagement campaigns.

### **RFM**

In [52]:
%%sql

with RFM_Base as 
(
select 
Customer_ID,
Customer_Name,
DATEDIFF(day,MAX(Order_Date),MAX(MAX(Order_Date) ) over()) AS Recency,
count(distinct Order_ID) Frequency,
cast(round(sum(sales),1) as decimal(18,2)) Monetary
from Superstore
group by
Customer_ID,
Customer_Name
),
RFM_Score as
(
select 
*,
NTILE(5)over(order by Recency desc) as R_score,
NTILE(5)over(order by Frequency asc) as F_score,
NTILE(5)over(order by Monetary asc) as M_score
from RFM_Base
)
select 
*,
CASE 
        -- Champions: أحدث عملاء وأكثرهم تكراراً لشراء
        WHEN R_Score >= 4 AND F_Score >= 4 THEN 'Champions'
        
        -- Potential Loyalists: اشتروا مؤخراً ومعدل تكرارهم متوسط أو واعد
        WHEN R_Score >= 3 AND F_Score < 4 THEN 'Potential Loyalists'
        
        -- At Risk: عملاء قدامى بتكرار شراء عالي لكن انقطعوا
        WHEN R_Score < 3 AND F_Score >= 3 THEN 'At Risk'
        
        -- Hibernating: انقطعوا من فترة طويلة وتكرارهم ضعيف (باقي الحالات)
        ELSE 'Hibernating'
    END AS Customer_Segment
from RFM_Score
ORDER BY R_Score DESC, F_Score DESC, M_Score DESC;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


Customer_ID,Customer_Name,Recency,Frequency,Monetary,R_score,F_score,M_score,Customer_Segment
GZ-14470,Gary Zandusky,7,9,4355.20,5,5,5,Champions
AI-10855,Arianne Irving,13,10,4375.80,5,5,5,Champions
TP-21130,Theone Pippenger,8,9,4454.10,5,5,5,Champions
RB-19465,Rick Bensley,9,12,4715.50,5,5,5,Champions
RA-19915,Russell Applegate,12,9,4793.50,5,5,5,Champions
DK-13225,Dean Katz,10,9,4802.40,5,5,5,Champions
MP-18175,Mike Pelletier,16,9,5087.90,5,5,5,Champions
EP-13915,Emily Phan,12,17,5478.10,5,5,5,Champions
MH-18115,Mick Hernandez,5,9,5503.10,5,5,5,Champions
SM-20950,Suzanne McNair,25,12,5563.40,5,5,5,Champions


#### **💡 Customer RFM Segmentation Insights**

**Customer Segment Breakdown (Total: 793 Customers):**

* **Champions (18.7% | 148 Customers):** Top-tier customers achieving maximum scores across Recency, Frequency, and Monetary metrics.
* **At Risk (17.3% | 137 Customers):** High-value historical buyers who haven't purchased recently and are at danger of churning.
* **Potential Loyalists (30.8% | 244 Customers):** Recent purchasers who show strong potential but still have moderate purchase frequency and overall spend.
* **Hibernating (33.3% | 264 Customers):** Inactive buyers with low engagement across Recency, Frequency, and Monetary scores.

**Segment Definitions:**

* **Champions:** High-value, loyal customers who buy frequently and purchased recently.
* **Potential Loyalists:** Recent buyers with promising engagement, ideal for upselling campaigns.
* **At Risk:** Valuable repeat customers who show declining activity and need urgent win-back strategies.
* **Hibernating:** Lapsed buyers with low overall order history, requiring low-cost re-engagement efforts.

### **Customer Lifetime Value & Profitability**

In [25]:
%%sql
SELECT 
    Customer_ID,
    Customer_Name,
    Segment,
    -- إجمالي المبيعات
    CAST(ROUND(SUM(Sales), 2) AS DECIMAL(18,2)) AS Total_Sales,
    -- إجمالي الأرباح
    CAST(ROUND(SUM(Profit), 2) AS DECIMAL(18,2)) AS Total_Profit,
    -- هامش الربح %
    CAST(ROUND((SUM(Profit) / NULLIF(SUM(Sales), 0)) * 100, 2) AS DECIMAL(18,2)) AS [Profit_Margin%],
    -- متوسط الخصم %
    CAST(ROUND(AVG(Discount) * 100, 2) AS DECIMAL(18,2)) AS [AVG_discounts%],
        -- حساب AOV (متوسط قيمة الأوردر)
    CAST(ROUND(SUM(Sales) / COUNT(DISTINCT Order_ID), 2) AS DECIMAL(18,2)) AS AOV
FROM Superstore
GROUP BY 
    Customer_ID,
    Customer_Name,
    Segment
ORDER BY total_sales asc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


Customer_ID,Customer_Name,Segment,Total_Sales,Total_Profit,Profit_Margin%,AVG_discounts%,AOV
TS-21085,Thais Sissman,Consumer,4.83,-3.32,-68.60,70.00,2.42
LD-16855,Lela Donovan,Corporate,5.30,0.46,8.75,20.00,5.30
CJ-11875,Carl Jackson,Corporate,16.52,1.65,10.00,20.00,16.52
MG-18205,Mitch Gastineau,Corporate,16.74,-1.25,-7.44,45.00,16.74
RS-19870,Roy Skaria,Home Office,22.33,9.58,42.92,6.67,11.16
SG-20890,Susan Gilcrest,Corporate,47.95,-3.71,-7.73,40.00,15.98
RE-19405,Ricardo Emerson,Consumer,48.36,6.05,12.50,20.00,48.36
LB-16735,Larry Blacks,Consumer,50.19,18.65,37.16,26.67,16.73
AS-10135,Adrian Shami,Home Office,58.82,21.85,37.15,6.67,29.41
JC-15340,Jasper Cacioppo,Consumer,71.26,-0.36,-0.50,22.50,17.82


#### **💡 Customer Lifetime Value (CLV) & Profitability Insights**

* **Top Revenue Customer (*Sean Miller* - Home Office):** Ranks **#1 in Total Sales** but generates a **negative profit margin (-7.9%)**, demonstrating that high revenue volume does not inherently guarantee profitability.
* **Lowest Profitability Customer (*Thais Sissman* - Consumer):** Generates extremely low sales volume while suffering a severe **-68.6% profit margin**, heavily driven by an aggressive **70% discount rate**.

## **5. 📌 Regional & Geographic Performance**
Analyzing sales volume, profitability, and average order values across Regions and States to identify key growth markets, regional demand patterns, and unprofitable territories

### **Regional Product Preference**


In [61]:
%%sql
select
region,
category,
cast(round(SUM(sales),1)as decimal(18,2)) total_sales,
cast(round(SUM(profit),1)as decimal(18,2)) total_profit,
cast(round(AVG(Discount)*100,1)as decimal(18,2)) avg_discount,
cast(round((SUM(profit)/SUM(sales))*100,1)as decimal(18,2)) profit_margin
from Superstore
group by 
region,
category
order by
region,
cast(round(SUM(sales),1)as decimal(18,2)) desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


region,category,total_sales,total_profit,avg_discount,profit_margin
Central,Technology,170416.30,33697.40,13.30,19.80
Central,Office Supplies,167026.40,8880.00,25.30,5.30
Central,Furniture,163797.20,-2871.00,29.70,-1.80
East,Technology,264974.00,47462.00,14.30,17.90
East,Furniture,208291.20,3046.20,15.40,1.50
East,Office Supplies,205516.10,41014.60,14.30,20.00
South,Technology,148771.90,19991.80,10.80,13.40
South,Office Supplies,125651.30,19986.40,16.70,15.90
South,Furniture,117298.70,6771.20,12.20,5.80
West,Furniture,252612.70,11505.00,13.10,4.60


#### **💡 Regional Product Preference Insights**

* **Technology Dominance:** Represents the top-performing category by sales revenue across the **Central, East, and South** regions.
* **West Region Outlier:** Shifts pattern with **Furniture** emerging as the highest sales generator in the **West** region, outpacing Technology.

### **Operational Efficiency & Shipping**

In [63]:
%%sql
/*Late Shipping Rate %*/
SELECT 
    Region,
    cast(round(SUM(sales),1)as decimal(18,2)) total_sales,
    cast(round(SUM(profit),1)as decimal(18,2)) total_profit,
    COUNT(DISTINCT Order_ID) AS total_orders,
    SUM(CASE WHEN DATEDIFF(day, Order_Date, Ship_Date) > 4 THEN 1 ELSE 0 END) AS delayed_orders,
    CAST(ROUND((SUM(CASE WHEN DATEDIFF(day, Order_Date, Ship_Date) > 4 THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT Order_ID)), 2) AS DECIMAL(18,2)) AS delay_rate_pct
FROM Superstore
GROUP BY region
ORDER BY delay_rate_pct DESC;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


Region,total_sales,total_profit,total_orders,delayed_orders,delay_rate_pct
Central,501239.90,39706.40,1175,966,82.21
West,725457.80,108418.40,1611,1288,79.95
East,678781.20,91522.80,1401,1096,78.23
South,391721.90,46749.40,822,643,78.22


#### **💡 Shipping & Operational Efficiency Insights**

* **Systemic Logistics Bottleneck:** All regions suffer from critical fulfillment delays, with order delay rates ranging consistently between **78% and 82%**.
* **Central Region Profit Drain:** High sales volume combined with the **highest delay rate (82.2%)** leads to severely compressed profit margins, showing how operational lags erode profitability.
* **West Region Resilience:** Ranks **#1 in Total Sales and Profit**, maintaining strong financial performance despite facing the second-highest delay rate (80.0%).
* **East & South Regional Baseline:** Demonstrate moderate sales and profit contributions while experiencing baseline fulfillment delays of ~78%.

### **State & City Profitability Analysis: High-Performing Outliers vs. Unprofitable Markets**

In [66]:
%%sql
select
top(5)
State,
city,
cast(round(SUM(sales),1)as decimal(18,2)) total_sales,
cast(round(SUM(profit),1)as decimal(18,2)) total_profit,
cast(round(AVG(Discount)*100,1)as decimal(18,2)) avg_discount,
cast(round((SUM(profit)/SUM(sales))*100,1)as decimal(18,2)) profit_margin
from Superstore
group by
State,
city
order by cast(round(SUM(sales),1)as decimal(18,2)) desc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


State,city,total_sales,total_profit,avg_discount,profit_margin
New York,New York City,256368.20,62037.00,5.60,24.20
California,Los Angeles,175851.30,30440.80,7.40,17.30
Washington,Seattle,119540.70,29156.10,6.50,24.40
California,San Francisco,112669.10,17507.40,6.70,15.50
Pennsylvania,Philadelphia,109077.00,-13837.80,32.70,-12.70


In [67]:
%%sql
select
top(5)
State,
city,
cast(round(SUM(sales),1)as decimal(18,2)) total_sales,
cast(round(SUM(profit),1)as decimal(18,2)) total_profit,
cast(round(AVG(Discount)*100,1)as decimal(18,2)) avg_discount,
cast(round((SUM(profit)/SUM(sales))*100,1)as decimal(18,2)) profit_margin
from Superstore
group by
State,
city
order by cast(round(SUM(sales),1)as decimal(18,2)) asc;

 * mssql+pyodbc://@DESKTOP-JEKI2AF/Sample_Superstore?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes
Done.


State,city,total_sales,total_profit,avg_discount,profit_margin
Texas,Abilene,1.40,-3.80,80.00,-270.00
Ohio,Elyria,1.80,-1.40,70.00,-76.70
Florida,Jupiter,2.10,0.20,20.00,7.50
Florida,Pensacola,2.20,-1.50,70.00,-66.70
Florida,Ormond Beach,2.80,-2.00,70.00,-70.00


#### **💡 State & City Profitability Insights**

* **Top Performers (East & West Anchors):**
* **New York:** Dominates overall performance, with **New York State & City** ranking as the top profit drivers.
* **California:** Shows strong regional strength, holding two top-tier positions via **Los Angeles (#2)** and **San Francisco (#4)**.
* **Philadelphia Outlier:** Ranks **#5 in Total Sales**, yet suffers from a **-12.7% profit margin**, revealing hidden unprofitability despite high revenue volume.


* **Bottom Performers & Margin Leaks:**
* **Abilene, Texas:** Represents the worst overall market performance with a severe **-270.0% profit margin**, directly caused by an aggressive **80.0% average discount**.
* **Florida Discounts Impact:** Features 3 out of the top 5 worst-performing cities:
* **Jupiter:** The only profitable market (**7.5% margin**) due to a low discount policy (**20%**).
* **Pensacola & Ormond Beach:** Suffer massive losses (**-66.7% and -70.0% margins**) driven by high **70% discount rates**.